## 🐍 Código para la Fase 3

# Celda 1: Importar y crear base de datos SQLite

In [1]:
import sqlite3
import pandas as pd
import os

# Ruta donde guardaremos la base de datos
db_path = 'data/olist.db'
conn = sqlite3.connect(db_path)

print(f"Base de datos creada en: {db_path}")

Base de datos creada en: data/olist.db


In [2]:
processed_path = 'data/processed/'

# Lista de archivos limpios y nombre de tabla deseado
clean_files = {
    'orders': 'orders_clean.csv',
    'order_items': 'order_items_clean.csv',
    'customers': 'customers_clean.csv',
    'products': 'products_clean.csv',
    'sellers': 'sellers_clean.csv',
    'payments': 'order_payments_clean.csv',
    'reviews': 'order_reviews_clean.csv'
}

for table_name, file_name in clean_files.items():
    file_path = os.path.join(processed_path, file_name)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Tabla '{table_name}' cargada con {len(df)} filas")
    else:
        print(f"Archivo no encontrado: {file_path}")

Tabla 'orders' cargada con 99441 filas
Tabla 'order_items' cargada con 112650 filas
Tabla 'customers' cargada con 99441 filas
Tabla 'products' cargada con 32951 filas
Tabla 'sellers' cargada con 3095 filas
Tabla 'payments' cargada con 103886 filas
Tabla 'reviews' cargada con 99224 filas


In [3]:
# Ver listado de tablas
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)
print("Tablas en la base de datos:")
print(tables)

# Ver esquema de una tabla ejemplo
print("\nEsquema de 'orders':")
schema = pd.read_sql("PRAGMA table_info(orders);", conn)
print(schema)

Tablas en la base de datos:
          name
0       orders
1  order_items
2    customers
3     products
4      sellers
5     payments
6      reviews

Esquema de 'orders':
   cid                           name  type  notnull dflt_value  pk
0    0                       order_id  TEXT        0       None   0
1    1                    customer_id  TEXT        0       None   0
2    2                   order_status  TEXT        0       None   0
3    3       order_purchase_timestamp  TEXT        0       None   0
4    4              order_approved_at  TEXT        0       None   0
5    5   order_delivered_carrier_date  TEXT        0       None   0
6    6  order_delivered_customer_date  TEXT        0       None   0
7    7  order_estimated_delivery_date  TEXT        0       None   0


In [4]:
top_products_by_state = pd.read_sql("""
    WITH product_state_sales AS (
        SELECT 
            c.customer_state,
            p.product_id,
            p.product_category_name,
            SUM(oi.price) AS total_sales,
            COUNT(oi.order_id) AS qty_sold
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN customers c ON o.customer_id = c.customer_id
        LEFT JOIN products p ON oi.product_id = p.product_id
        GROUP BY c.customer_state, p.product_id, p.product_category_name
    ),
    ranked AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY customer_state ORDER BY total_sales DESC) AS rank
        FROM product_state_sales
    )
    SELECT 
        customer_state,
        product_id,
        product_category_name,
        total_sales,
        qty_sold,
        rank
    FROM ranked
    WHERE rank <= 5
    ORDER BY customer_state, rank
""", conn)

top_products_by_state.to_csv('data/processed/top_products_by_state.csv', index=False)
print("Guardado: top_products_by_state.csv")

Guardado: top_products_by_state.csv


In [5]:
logistics_by_state = pd.read_sql("""
    WITH delivery_metrics AS (
        SELECT 
            o.order_id,
            o.customer_id,
            o.order_purchase_timestamp,
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date,
            julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS actual_delivery_days,
            julianday(o.order_estimated_delivery_date) - julianday(o.order_purchase_timestamp) AS estimated_days,
            CASE 
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 
                ELSE 0 
            END AS is_delayed
        FROM orders o
        WHERE o.order_delivered_customer_date IS NOT NULL
    )
    SELECT 
        c.customer_state,
        COUNT(*) AS total_deliveries,
        ROUND(AVG(actual_delivery_days), 2) AS avg_delivery_days,
        ROUND(AVG(estimated_days), 2) AS avg_estimated_days,
        ROUND(AVG(actual_delivery_days - estimated_days), 2) AS avg_delay,
        ROUND(SUM(is_delayed) * 100.0 / COUNT(*), 2) AS delay_percentage
    FROM delivery_metrics d
    JOIN customers c ON d.customer_id = c.customer_id
    GROUP BY c.customer_state
    ORDER BY delay_percentage DESC
""", conn)

logistics_by_state.to_csv('data/processed/logistics_by_state.csv', index=False)
print("Guardado: logistics_by_state.csv")

Guardado: logistics_by_state.csv


In [6]:
ltv_top100 = pd.read_sql("""
    WITH customer_revenue AS (
        SELECT 
            o.customer_id,
            SUM(oi.price) AS total_revenue,
            COUNT(DISTINCT o.order_id) AS num_orders,
            MIN(o.order_purchase_timestamp) AS first_purchase,
            MAX(o.order_purchase_timestamp) AS last_purchase
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY o.customer_id
    )
    SELECT 
        customer_id,
        total_revenue,
        num_orders,
        ROUND(total_revenue / num_orders, 2) AS avg_ticket,
        julianday(last_purchase) - julianday(first_purchase) AS customer_lifetime_days,
        NTILE(4) OVER (ORDER BY total_revenue DESC) AS revenue_quartile
    FROM customer_revenue
    ORDER BY total_revenue DESC
    LIMIT 100
""", conn)

ltv_top100.to_csv('data/processed/ltv_top100_customers.csv', index=False)
print("Guardado: ltv_top100_customers.csv")

Guardado: ltv_top100_customers.csv


In [7]:
delay_vs_score = pd.read_sql("""
    SELECT 
        r.review_score,
        AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)) AS avg_delay_days,
        COUNT(*) AS num_reviews
    FROM reviews r
    JOIN orders o ON r.order_id = o.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
    GROUP BY r.review_score
    ORDER BY r.review_score
""", conn)

delay_vs_score.to_csv('data/processed/delay_vs_score.csv', index=False)
print("Guardado: delay_vs_score.csv")

Guardado: delay_vs_score.csv


In [8]:
conn.close()
print("Base de datos cerrada.")

# Listar todos los archivos generados en processed
import os
print("\nArchivos generados en data/processed/:")
for f in os.listdir('data/processed/'):
    if '_sql.csv' in f or 'top_products' in f or 'logistics' in f or 'ltv_' in f or 'delay_vs_score' in f:
        print(f" - {f}")

Base de datos cerrada.

Archivos generados en data/processed/:
 - delay_vs_score.csv
 - logistics_by_state.csv
 - ltv_top100_customers.csv
 - top_products_by_state.csv
